# M49 --- external validation of the best ICBHI model on **HF_Lung_V1**

The best checkpoint on the corrected ICBHI official split is run, **unchanged**, over a
corpus it has never seen, and scored with the same official ICBHI metric. No fine-tuning,
no threshold tuning, no target label used to fit anything in the headline number.

**HF_Lung_V1** --- 9,765 fifteen-second recordings from Taiwan, natively at
4 kHz, labelled with breath phases and adventitious-sound spans. The shift against ICBHI is
population, hardware **and** bandwidth: 4 kHz sampling puts Nyquist at 2,000 Hz, exactly the
model's mel `fmax`, so the top of the band arrives attenuated by the source anti-aliasing.
That is a real confound and it is recorded in `dataset_info.native_sample_rates_hz`.

**Unit of analysis.** HF_Lung_V1 annotates breath *phases*, not cycles, so each inhalation is
paired with the exhalation that follows it within 1 s to form one ICBHI-style cycle. An
unpaired phase becomes a cycle on its own rather than being dropped --- discarding lone phases
would throw away the breaths at the clip boundaries, which is where a 15 s window cuts a cycle
in half, and that loss is not random with respect to the label.

**Taxonomy.** A cycle is Crackle if a `D` span overlaps it by >= 50 ms, Wheeze if a
`Wheeze` / `Stridor` / `Rhonchi` span does, Both if both, Normal otherwise --- the same
"contains the sound" rule ICBHI uses. `I` and `E` build the cycle and are not labels. An
unmapped token raises.

**Confidence interval:** **recording-level**, not patient-level. HF_Lung_V1 filenames carry a
timestamp and no patient ID, so the bootstrap resamples recording sessions (all
`trunc_...-LX_N` slices of one session stay together). Calling it patient-level would
overstate what the resampling controls for, and the results JSON names the unit explicitly.

## Before you press Run All

| | |
|---|---|
| **Accelerator** | GPU T4 (CPU works but the pass is slower) |
| **Internet** | **ON** --- the notebook downloads HF_Lung_V1 from GitLab |
| **Add Data** | `vbookshelf/respiratory-sound-database` (ICBHI --- needed for the verification gate) |
| **Add Data** | the checkpoint: upload `Asif's/M22_v2/Results/best_model.pth` as a private Kaggle dataset |
| **Runtime** | roughly 10 min download + 10 min evaluation on a T4 |

### The gate

Before a single external number is computed, the notebook re-scores the checkpoint on the
2,636 ICBHI test cycles and **asserts it reproduces the score stored inside the checkpoint**
(M22_v2: 0.5602). That proves this notebook's preprocessing, channel expansion and ImageNet
normalisation are the training run's --- so a low external score can be read as domain shift
rather than as a bug in the harness. A mismatch aborts the notebook.

### Bring back

`results_M49_*.json`, `preds_M49_*.npy`, `confusion_M49_*.png` --- zipped in the last cell.
Commit them into `M49_cross_dataset/`.


## 1. Environment

In [1]:
!pip -q install librosa soundfile
import glob, json, os, subprocess, sys, time
import numpy as np
print(sys.version)
import torch, librosa
print("torch", torch.__version__, "| librosa", librosa.__version__, "| cuda:",
      torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

WORK = "/kaggle/working/M49"
os.makedirs(WORK, exist_ok=True)

# Set to a row count for a 2-minute smoke run; None for the real thing.
SMOKE = None
# The frozen-feature probe uses target labels, so it is NOT a zero-shot number. It is
# reported in its own block and answers a question the headline number cannot: whether the
# representation is useless here, or only the ICBHI-fitted decision boundary is.
RUN_PROBE = True


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch 2.10.0+cu128 | librosa 0.11.0 | cuda: True Tesla T4


## 2. Locate the checkpoint and ICBHI

By glob, not by a hard-coded path --- Kaggle's dataset mirrors differ in layout, and a wrong path that silently resolves to an empty directory is a failure mode this project has already been bitten by.

In [2]:
def find_one(pattern, what, hint=""):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"could not find {what} with {pattern!r}. {hint}")
    return hits[0]

# M22_v2 (0.5602) is the model the paper reports as best, so it is the one this external
# validation is about. M45 P3 scores higher (0.5764) and the paper explicitly declines it:
# the delta is 1.1x the seed noise range (0.0141) and its patient-level interval spans zero.
# Any M45 / M22_v2 checkpoint loads here -- the loader reads the preprocessing flags out of
# the checkpoint's own cfg and the gate holds it to the score IT reports -- but swapping this
# to P3 puts the notebook at odds with the paper's ablation section.
#
# NOT best_model_official.pth: same folder, 0.5641, trained on the published split verbatim,
# which leaks patients 156 and 218 across train and test.
CKPT_NAME = "best_model.pth"
CKPT = find_one(f"/kaggle/input/**/{CKPT_NAME}", "the checkpoint",
                f"Upload Asif's/M22_v2/Results/{CKPT_NAME} as a private Kaggle dataset and "
                "attach it, or set CKPT_NAME to whichever checkpoint you attached.")
ICBHI_AUDIO = os.path.dirname(find_one(
    "/kaggle/input/**/audio_and_txt_files/*.wav", "the ICBHI audio",
    "Add Data -> vbookshelf/respiratory-sound-database."))
ICBHI_SPLIT = find_one("/kaggle/input/**/ICBHI_challenge_train_test.txt",
                       "the official ICBHI split file",
                       "It ships with the same dataset. A notebook that cannot find it "
                       "must raise, never fall back (Model_Training_Protocol.md 1.1).")
print("checkpoint  :", CKPT)
print("ICBHI audio :", ICBHI_AUDIO, len(glob.glob(ICBHI_AUDIO + "/*.wav")), "wavs")
print("ICBHI split :", ICBHI_SPLIT)


checkpoint  : /kaggle/input/datasets/sudamchandrabasak/m22-checkpoints/best_model.pth
ICBHI audio : /kaggle/input/datasets/vbookshelf/respiratory-sound-database/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files 920 wavs
ICBHI split : /kaggle/input/datasets/sudamchandrabasak/icbhi-challenge-train-test-split/ICBHI_challenge_train_test.txt


## 2b. Get HF_Lung_V1

The corpus ships as split 7-Zip archives on GitLab (`train.7z.001`--`010`,
`test.7z.001`--`003`, about 1.1 GB total). Tried in order: an attached Kaggle dataset, then a
direct download. Needs **Internet ON**.

Set `HFL_PARTS = ("test",)` for a faster first pass over the test half only --- but then say
so, because it is a different corpus subset from the full run.

In [3]:
HFL_PARTS = ("train", "test")     # ("test",) for a quicker first pass
HFL_ROOT = None

hits = sorted(glob.glob("/kaggle/input/**/*_label.txt", recursive=True))
if hits:
    # commonpath over every label file, so a mirror that flattened train/ and test/ into one
    # directory resolves just as well as one that kept them
    HFL_ROOT = os.path.commonpath([os.path.dirname(p) for p in hits])
    print("using the attached dataset:", HFL_ROOT, len(hits), "label files")
else:
    if subprocess.call(["which", "7z"]) != 0:
        subprocess.call(["apt-get", "-qq", "install", "-y", "p7zip-full"])
    assert subprocess.call(["which", "7z"]) == 0, (
        "7z is not available and could not be installed. Extract HF_Lung_V1 locally, "
        "upload it as a Kaggle dataset, and attach it - the cell above will find it.")
    HFL_ROOT = "/kaggle/working/HF_Lung_V1"
    os.makedirs(HFL_ROOT, exist_ok=True)
    base = "https://gitlab.com/techsupportHF/HF_Lung_V1/-/raw/master"
    n_parts = {"train": 10, "test": 3}
    for part in HFL_PARTS:
        if os.path.isdir(os.path.join(HFL_ROOT, part)):
            print(f"{part}/ already extracted, skipping")
            continue
        t0 = time.time()
        for i in range(1, n_parts[part] + 1):
            name = f"{part}.7z.{i:03d}"
            dst = os.path.join(HFL_ROOT, name)
            if not os.path.exists(dst):
                subprocess.check_call(["wget", "-q", "-O", dst,
                                       f"{base}/{name}?inline=false"])
            print(f"  {name} {os.path.getsize(dst)/1e6:.0f} MB")
        subprocess.check_call(["7z", "x", "-y", f"-o{HFL_ROOT}",
                               os.path.join(HFL_ROOT, f"{part}.7z.001")],
                              stdout=subprocess.DEVNULL)
        for f in glob.glob(os.path.join(HFL_ROOT, f"{part}.7z.*")):
            os.remove(f)
        print(f"  {part} ready in {(time.time()-t0)/60:.1f} min")

wavs = glob.glob(os.path.join(HFL_ROOT, "**", "*.wav"), recursive=True)
labs = glob.glob(os.path.join(HFL_ROOT, "**", "*_label.txt"), recursive=True)
assert wavs and labs, (f"nothing under {HFL_ROOT}. Turn Internet ON, or attach an "
                       "HF_Lung_V1 mirror as a dataset.")
print("HF_Lung_V1 root:", HFL_ROOT, "|", len(wavs), "wav |", len(labs), "label files")
print("a label file, verbatim:")
print(open(sorted(labs)[0]).read()[:300])


/usr/bin/7z
/usr/bin/7z
  train.7z.001 94 MB
  train.7z.002 94 MB
  train.7z.003 94 MB
  train.7z.004 94 MB
  train.7z.005 94 MB
  train.7z.006 94 MB
  train.7z.007 94 MB
  train.7z.008 94 MB
  train.7z.009 94 MB
  train.7z.010 90 MB
  train ready in 1.0 min
  test.7z.001 94 MB
  test.7z.002 94 MB
  test.7z.003 46 MB
  test ready in 0.1 min
HF_Lung_V1 root: /kaggle/working/HF_Lung_V1 | 9765 wav | 9765 label files
a label file, verbatim:
I 00:00:01.120 00:00:01.937
D 00:00:01.120 00:00:01.937
I 00:00:03.340 00:00:04.157
D 00:00:03.340 00:00:04.157
I 00:00:05.674 00:00:06.491
D 00:00:05.674 00:00:06.491
I 00:00:07.855 00:00:08.805
D 00:00:07.855 00:00:08.805
I 00:00:10.018 00:00:10.968
D 00:00:10.018 00:00:10.968
I 00:00:12.105 00:00


## 3. The two modules

`m45_ablation.py` is embedded **verbatim** --- it is the module that trained this checkpoint,
and it supplies the mel parameters, the official metric and the corrected-split loader.
`m49_xval.py` imports it rather than re-implementing any of it, because a second
implementation of the mel parameters would make every cross-dataset delta a measurement of
the gap between two scripts.

Both are carried as gzip+base64 and decoded to /kaggle/working, byte-identical to the repo.
The two cells below therefore look like blobs rather than source. That is deliberate: they
used to be triple-quoted literals holding the modules verbatim, and a literal that ends one
character early runs the rest of the module as *cell* code, which surfaces as a NameError on
`__file__`. Base64 has no quotes and no newlines, so it cannot end early.

In [4]:
M45_ABLATION_B64 = (
'''H4sIAAAAAAAC/7Vde3PbOJL/P1X5Dlim5kwmEiPJj02c1Vw5iTPj2jx8tmdn9zQqhpIgi2tK5JCU
bY3XW/ch7hPeJ7lfNwASpORHZu9cE5svNBqNRqNf6HEc5+mTTzu74n/+67/Fu2SeJgu5KMQLkWYy
zZKxzPNocS7CURwWUbIQ+K+YSTGSeSHmyUTGwj05+7PI5K/LKJNztM1FtyMOPr8XXe/pk6dP3h6c
Hn48+ny4Lz71esFljzv6lIyiWH6WxV966Oo0leOD5Tk1bolkOo3GURiLo3dvfzwSHX93r9MTA/zt
vOq2cL/X3f7jEHg8ffLVfBvsdYKdTpACRcAIosVEphK/cD1OskyOCzn56osvCykuwyzCYKQYz8LF
ucxFKjORJVdvhLyU2aqYYbRPn8g4lyLKxUzGExEWPGSNPqBFqfRpaD//+Dfx5fOhOH13cnR8Jj58
ORFnP38RJ4f/8dPRyeGnw89np0+fCPyAQoFNIX8+ETmwIoKCWHm4ykWeiElEqMarfeF8ktk5usS8
hIsJ/u6JaFEkGLUsp0KBNmAiNS+YtCQr3tD1SoSZ5Id5OJfiAlQBcTHMCIQZS9+hIZz9eCjOfjw5
PBSfjk5Pjz7/IE7PDn44PBUHJ4fi85efxcmXn09b4u1PZ/js8G/i4P17cXKASwz1x4PPGOunL385
LEdZorM9EnGUgxfy6LrBSiB27ouzWSYl4QME52pMQLQIaUqKGUg+iSZikRQKtLwGMDVGzEqKCYij
hdwHcfAqjcEEhRiBUu00zHMxjeJCZi2BcSYR9dliKmKm2+PVOJYKZjhHw2I5kegmm4dAl8nqi1Mz
hlCE+QXAJRn6TfIKvwT8L66yqCjkAr1lecH8QFDPDOEXyZWI0AXPuJwI9yshSPh9bYmvCjVJl8CD
EPhqpnCcLKbRuadwNvOogH/58IG+oiVFFASVQA2swWgkM9APz5bZgpfrYlXSVjEHtRFXyRL8rDhf
HHS4C01gYn4CVYQlmQteJ7iYyzBfZhhDeB5GC0zEQYdXcYjZjUlajMKcJ8SA5n4U4GhxCdJOaHaZ
FdHRgllAhHEmw8lKZMuFGapIQ1qOquPREtMoplky53GUFD5NxHG3fbzNVCZ+JGgt4hXNjfxgX8hw
PBMFCJLzsuG5I2JQX2qZ5OKKaaiZbLRcgTFPgBR1GGU2OdAyTZM8wjDUKiUGAR8cdNsHewzyeKd9
vOuLA8GfRZdSNVeguVuDLJE0JDmSzMEbk9DACidJqkQN4/oGHy3keciwaARoZhYvLxEFuvye3hMR
5pFaZ5py1noRI3AFJjqPzmeYtGVBD+YSfMzEPaDFTstcnB4fvhPvj95DApyJg9M/k2hrEfqH7yHw
xOFfjt4ffn6nl/3xTvCbzJI0nPgQVDsWtxRFFqEXGh6YOscSBu5g53GW5DkjfpWFKdbsZCJ5TmQR
0deYsKlmS5pDXrNM42myxG+s9RhUdNERWKnf8TuvX70BheKV2O59R3zPLUDkEUbLX+x6vjhkFudd
RQFXcEGffAZmIL4D9zJar0SOaU9BIHQyifC05+/0RI5FCTHNjGpIWkTUE1D65/a1uIqKmYL9dZH6
9OqrXiizMJsQEcZYCAs948R7sQwvCVNwVY7XRbRYRsWKthy1InMZVpz/DghHY0H0ogkGyzCiOc0/
+Ag9pFkI1hwzn/NINcJRbrgDY4kmUZKvFmOIjGisSTFL0OoN84QG355kEa1VJs54yagq4cLsO0nG
Sy3ZpmEUY8JZHaDVEi7RgYYbQ95F2KR5yEqk5Ql2o3Q5gsCdoXUpI0M19UQ18fPR2Y/oZBoyxJcJ
oUUbL8TMCuPWAvwyiSZMIYiCQk4xdN5YgB+tskLmvMIl5ivT0kURLh9DnvDMTxK6v0qyC/pAgx0n
yxyUYb4geQlZz9/54ogFLksDTD11Mk+wPEOiApZSaHifhiNA3plhqtFKwwal5fiCtkaeVdpd1RuI
1EqMWj99o3e4lrxsYbEscznxdNuuEG1bi6rablPTe9v2qO3RHAIE6hiIHanWGaYK8oPuo3Jv1E22
qQlPrbiSJEuIiEKkMXYHsVyoZyCJkRAHO4Kk0W/gplE4vhgleohFFpo9T/MJ6EXrWDfbFWKP5Emc
2xTBWop5awNM6FQsUjKZJ/HSQnFPiB2sDS0I1psWEppuBi2TFp28LowoE4JkWdtwiv7BLa9ttd+N
iSchy0LWp8CvvBhpFJVycbxLNJpHi/Y8vGYFQ0GaQFAqXoQykmPGiiw5z8DBg06rO6RxjEOC8fTJ
T6fQwxSwFGopFvR8Zzcwyp+frkS7TRxIE9j4eWbY84HmYRyLDT/PtOxZLmhrBhgIESY0cYKZnDuB
5sv5HDp2LptAaT9Xu7za33l/4g2Ytbv86ROHbBF+HATTJbZuGQQkHyCAIDkgvZSIIOqYp9l5Gma5
LB+QolFE8+rBeZyMypu/58Qg+mYekrDWN0leXqrm5e1iOce4QkjPlJ4+E1+BHNggCL6SMMCGJKdY
sxOSNFpxypNlNlb79bUcb01YKJL0laMkuRBjCbpbHDSSbHM800OVE7JUxJ/D8/NYKU1lS1I5WT8m
pSyZLGn7IrPg5QV//JIEGau7V7MIGycJ6jhPCDRvtuqtVmCSbMWbw/hqYoRaxnpBuMivlMCETLeF
GDQFEr/Q+sIpS0CIMBgCh5AzbpL7sL5mPmAvYG2U9+Eop7+uoZnniWgqHHPrEGloioCnqyUS7/tk
fwHGuSyAn+vh1cnh8Rf01ARs7v8OCe0SNi3h+L6jflO7dx8PTk9h0PTFwPnMij69fIeN8iKWdPnz
TMrf+OptUsyc4dMnHw/eHn4MvnxAmxsXWnLH2xf443bVJf7Q0y4ue+opXW7fPn1yeoIm3b1Op2Ms
X9xPonHhLgISY/1u7xXMkmXGjBzk/Vc+AGlh03dIGwIekBoQGv2zbClbFUlYQ9UWRP8D6CWNgSPN
rbYk9G29KbZPFrdyogCT8MSwTVMWwIGR3PoTkk/h8nwTJnHW35XtnRZQKsazII8AqrvXEtCrx7O8
v4NhkajDxg3ta5sECKDu9Dyz6RFpFUTnoOfsKyJZOGq0ggkEYt+xNih80+aPiJNda5fyHK9lIG4b
iI1hNaHy63a5X8WklrpqG3t3aAPcMQA11RR9NJzG1tZq7mYWnF0DR/PD3k4Jxex0lmAAt1ht90xb
i392iH80AGu/s2BAQ6xgHJfjKHmOtjunhFHb/CwoY6V5qr3OgleOR/Nsk8DNXc7eEEutooT3rDTn
sHeCY1goQdHMYUeQ+vWe7THSZPL7rTIWZmxjkgXgG2y7BttyHdXm8cWaC0G4u512D8tZ/PibeLuE
uZ9BiBYzizWOS+41i7EBUw0/jNtkyZH0NU4JC0bJr2YFN2CUngtchRd3eS4YIKQQiBiYNUaDbjGW
Le7nlgTpT6ew46wF2EH3dUFKwhYNDsDFWzk1VUooXZ3IfBkX/DBTl4F66dPuWg5K/zilTuuqryAp
73T+WczefQRK2/dgtH0nQnVF2eUvPUM62LvBwfHxx6N3B28/kvC+uWW+VA6HK6kdamiZRZLcg5m0
fG0l95LvxyUGLpmXBR9txO1/8Yc0HDgVoHSI0rUZ5MQP7Pa8dtn6CrAcSH7TY9ppvX3tLaQnPCx1
T64tnh0spwQuU3e9CSvqaEKf+bDR43AsXeeXgsgtHK961DJPGIa9m2PHh9rsFp74vi96ytkz6A59
WPEycz3q3HVYqBMIstscu/cS8UEx6AyrDv2r8JIaON4Q+NkQtbGV5zIruGtu7ol+X7zuQWJOnZvq
6a34Az9mj0xGki93tI4L8xIDD+ejCZSefdK13FwPzwkcD9jornK4ZfIGXeGTuqSh8fc+ZNQcKo41
LG7j57LAXIbgWhfduexikKCe50MKu5caPnlt4jClHlIGnhrgDEQDN4S+BKFF97Yx5RiAoTJ9qHrj
idfAWelC2zuRv63RFYq+2+WPrS/h7FtKjcol0Vv3yLTf3YWu5JRsq1G7yhIIQ+Pj4PUDXW1YERLT
zB2wbuyStujTr7p4sNjeec6c4Xk1amMIlvrIXcPuK4GQnGLNFU09mlmL+68Lq2WzNwb8AuomPnPq
TM+vyOliyCMwGro3sJTh46Jlk9+1L0hWT9eW6oZWqVmna0vQWoYpL8OdRlNDez9MKXTi3jhEw30i
PihKI8EN/cFdGWeZ4BnzEZ435Oz6D4DATYMW0zgJwe20egAMnVnPusOH4cThSMZoY9T0gUsLMx30
CJ663B563vBhhIhKNCwWLjSI4a2mGTRReI6ZIkpyk8AdsQYQGO3BDVtQGfu7HVLCZlGfFAW6hBCR
WX/HTA6s2v9ktWoGFhM7xazNH9j6hKV4uLUoSqhduz2vdAKSZkL/lMc7VorKPnmWwmUOh4bWXPJZ
NIX7C6sPjleOypHGBNNWOeHGyv7Jq9iC9eUVBzvwsfb2sc1OgTFlqpYeXr8coEaNTPcc4bGVDxfz
AshoG1oRrlXirj5frH4Fv8Jmegn3akc9m0V4hH7dWdTiD54j4vf69Ws9LSNgH+ALBdBlQrbEIIYJ
TF/TNKirISyTYpXKvkOkdeqzatBwGRz+Qdjm9LULry1z4nbPq6addK7AaFrSDa2JPa4rZjV17I6p
3PWg3MJFmJDfdwKHPjwFFFbxSzKSnFLrAehAW6Y/MHfRsVcfhxtisPNNyNNan9MmINuvlFwPq/EY
jTSARirdlLbMFnWYZEE6Lvq9XWZibAp9TIs12FOjyebLUcGOZoyrVGdNZJjhCVvp30yHPc+OmQnW
nRUW7DiBB3bOflUKCFWOvhG5Usj/TBEAvBqTVwrkJ3Q1e7Pvo9yp4KTOtRMlUjHdXLmRQni4yKfH
vpkqxECfhpNLAouvluzMWUy091wz/Kkev9RewVDtoTW6kDcaQ+GoBQ2K4yHsXKz5VPAMBAHTxDrg
kcLFSJ5yeGZCHm2hw3zpMjODGFHoSOYm+lZzJeENBwBtoGVECr6skfaFxwn84AUYCQ7pnG1X5Uq6
xGSSdw7vlmNywdVmyaK4IZYZKKnGEUZZJOfK404fsMYMfYa84yr8F0/bpOAp6WXc4gU4GNHLU4tp
FBvx5FgTRgRh9/7yfAZXFGK/nK4Av2QZmentmhiMhSzGjwlEAAPSpPdah0ZJji3elJyuPF7YZvG5
nkhmNER5DYOR+7iKuqF3pcfwBCP0h2B0mMVk9mPIbOy53Q74o0JEr6suAlJaBkzYg5iJ3e/KoFdU
KNpx8GGG+BAJc+zKE8X5avkwWyhBDG/qyoTLZ2woHy9zFtcUdSIq5iUfqMnl0B+inWK+zBGbiRVf
GbdhKN59/oztZKFCPtQrqUrghSqaBCRWGmoVfwYaFOeT2SUHLZvbg2LePhypfkWRNfkDcQytqA9N
8ULKdBLNlZVel3xKMEZzqKCKUdpqgp6bkWAjtiQ4mJ38LS4rM6yGwFO1mMBTMz23BNyZyvKo5yu0
1BpXLnF2dYdqfdAs5HiHmDceVOJb73kgCsKcOv6r3D3s+gMB0OvAUY+cYUvdVk4dZ2ja5GTos11B
Wg02yucEwDOvp1gn/LYL/bP6+uVLckGaTX1l6XnQVwLWEBkzGEvhRJMk65+ecOoNGLevCWQw6tMW
BGqRT0a9AXF3H1LT5skisSdOXo8lotuH/IcEJLhKWrjB92PkGeRWmwWlvYmwN/rKxOO0m1CH820o
iKJmZqXAUxQr0e6LDwhS2pISSxxbTLEs2PdetXc/UVJTcKZdjMFxlhTJOIlrKTs+62Ol2hwSX58s
F7T/HGZZkrkOxURZGAqisgqJwghiDUkrXFodD9k06tRIcThPC8rGGHNENedIKMMj35cSnWXuBVNK
DfJelKaOZKiMioY9UQjdgAluHc8IeiAGliTfu+sYXbdmkIelBmbpwnZzPa4/VVy5XzOMmOO1m9EZ
spnIvu/9O5jpmYoKlzE24bJnUNnNDSsnVBKGZQsYnqI8/lhGsWutEI0g7Lz98umwHn3YF9/082wt
RrgRK3xESFH4oEKnXaKjlwp1X6d2WEd0fZ6043BtmtZ0102Nteey1viZ+IFydrSblYxKbJ6nZx/O
4NOaY2206PkCojL5O1YFbnhP1tuqsULAHRdmY64tUktRmrHBmZssuyqSa2ltV6GtEYaFxekkIxRt
SUU2oi0vpgURGkJyWvS7nR787LMkDUDo82KGSEWHxPrC3O90OghMPX8uehXguSUppzJkVYgkdiWT
3NN+XZWmO6+UptrFr0X/A7Zo/cdGewp9DjYmXcDBTjbmZjZ5BL6rfngHdnaP34Tp/UStsP8moOVI
W0oNZDtEQYjtYfLLoIAjZuTOKa9h2leKgWfz+MBRwQlnaFGL4bj4jcU398ngBAO8pCdscXnWc2yu
ZEZtprrCZ46mr7Ar0qf4/ZCoIMVGTN6WaiwWGYf/tZ7NEP0cC0MOzEPuRwuQGGN1VVzSiBKtCrTF
macirGcsfPVjNgDj+WC/JfbNw2FNn8LLz9Ckh/fawRzAD8ZIyZKuyrkDdSEHwlKHeqaTrCwV1aTZ
qqSpa4pwQeksaBPXsuVCrihtjuHCfi94UycQMb8zkNXeX6WMLOQVZ+BoQZHgJW63cltryK1kP1ar
lVZAMQItQtABMQIcwsQpW2o5bA1v54F6UKlleJjrh3rnwpPghjQzfqiYbGvo3To1Zp86wSg1n7G8
3TKb5lZLcMDM826DyaL+jZbJ1idrYMNGE70H2E0264qlNrmmeXo17ZF8lE3Ppw6x82w57EnHXIMO
IOStv0hXpks0mocXSJ7L8rXkALoB67L3M0gubDWRuZ6XJrZEYjJ81201NWYznmqdNzyq3IO1TC2z
QSI9JeUPoOKyi8gwO8WvKYet72QYGWPS59+6mzQjwk0dlSuKtUAmlmLbmzU3MmNwK3zKQtBuHb2E
H4XB1YuNKJATGI4xNowlrHk2uxSd7K1gPogoJGLMnmzAzlwQDlfKC6uuyfeqDJA1J1J3r+7LdiMS
bp74TnR3EQ9tqKxN8ghxE73o3r68KafxtiIDOoB96q5bdL9vav6P4mpz5B4g6U+JOnOwwB3PDWHH
evrCPMyycIU3LeWlM5xLzvfxfEByeYj9gC6HPgVJWCDbtyyPlYfPWYQLQ5lwtFAguvv6S5PiTwuC
nmMtDDENuETiSU9fIrtimxYut0dP9OeOHoz/k18gQCkoPE1Ne54eDB5Wl6ntBNU+/3HkIm8IFjLs
ELj+aS0iNanQ27VJ89i3HcIX7EbwNYGNfUxp7vA+QO5Aq8uia9Vi+aui8nIRweNHITHvjcARAgp8
7Qtmh7BYJAuyelyOz0HHVSGrlBbF8lcdpMqwNhmSShHxtbEQ4DlFZgnsZSPOFBCAjLYrVw2qFj9S
0KoUXukOoutBOqz6BmhfJdK6S7iiifWXv3qkl3DAUom5obWsaFgVpzXo4a4G+VBRmi846pH3ib/A
Bz2e9HqcCnN/TfS4bizMSxPTua6zwSAj16arJvs3ILqjCPkbDabupIGPYNDzd1vi9R/93aE3/D9b
dZRhWCYrEp/hQYAbEhoBsZdSNJDnV0o4k6aXZGPICf7jL9ifsFjoe3JimlRQlQbPX8GsiHOfAugG
yHtcw+XR4ouPsNRl9u2ci/XSDdjvAETHY2yn45W6N8YWBKxJBKOkMPAebdrLlLIV3ZuLfbAiEf5C
B10peWOgCDC0Y7tk91/4LMDJ8TTjILSJWFGuCDqx2w4cziAxDiUEG/CBM15OQo4AK6LQrR/lQXgJ
FwN5soyAcsbp0vnX9Ae9I/yyuNnqbz3/487tLwvsDAq7231xQ9jdVm8rF0RBbrJBxlThzY7DwsCZ
NjCO2g2t4LIeXyEf2YgSDHSbvxbwPf6VW9rqLT1l5daEr1v117J8zckKCtaKsaYtgjeIAbpVocph
hVKRDTH7q0I+/KU0okKlIdS+tiKwa03MZKtAT4AEPiz8/Xog2RJ1vWYQear9jYo9SHzSHcROb5ez
HZGaQDzpeuvh5Q6sjin+XZfWy4Yo87RzVwdVM1gyU9I3Gh2+EddkxEw7+9POiymsGdJyOo8fWnFX
z6+6jxlaYQ2tt2loxYND69HQiruHhv+Kzn7ReVE0hqZl9nUpUjj9/v2pq0WYPVSa+yCgpMggcCE9
/9oS2LPBCEi+yn3c5T7l/vp4gk7Kt00A2MCoPRoZK5FyZvyV1/zwnM7tyLnqLGoSnXY5RQ8SqgHn
U7uWIgWMoK62hGVzNshvwp0VP/tjBMmkq2xdNRCWWdee2RGwRyNhxFUpOyt04FVLIw54pVYy3wUd
WRKs9MIngcG5rw7lFFg5rvyuunfuSTLIZ8vpNDb5gBh2QAnYOH3VN84MQkRuQEQSItKYcQ/0vwZZ
q9l0OKpfJ8bA33mFNezv7O7R787e0PMvI3lFKczbrFZ0jcpZTNYb93qv0axHTin83t3Y2GZOZPi5
i4X/iRPV7+bPtVSvZUo5XH75RYMbrmgTO/qE0xGfD8+6fw7+0nUqn3KZQQy5yBxBbo0GfOMey80Q
lbrg83FmmK2cnriQBRIDXZUbnPevvLLVmmjgnlVisLNR4BkFseoZxiJtqHCR1nPCavaUr48M5wF8
GTQfzA/NwZxzWhjofDAJUwq7HVyeHydJ3Ju4KjUdogVKj0rGVl++Vzeu2r7VjTP0mpCtNGZu9pGd
wC6ykjukKxJYPtjSX6d9fa5BgCtEIkk6XG+SDteU0wfKrPOhnUzFnW2gFUFwr+m0DVjeLxJIB5VW
wR49cHLt2UbJYo/VLYnlMnHdatawj3psgGBJuF2vkihqoczDxRKuYLIuFGnpyqH9vjJD1l7q1con
6fu8ZDzCF9g2nJj1VPYao2kHuImI4hJxcpxig+xjgQbXlPHJlna+Ml3rlKaVNdY2KnyhO6ThAXt6
fIUHVz5R2F6O4yzSLPWOjnQeIi02SVcfcakXTr8mQq6MWa+3AyXtWzoLpk9j3uxlvacf425Ki1Jg
4Tqa03KYu4O0Wn1M49rC4zTI+ip7IGkMxx54MuKMJK8aI3Tscbjqw0e8Y6TneNZAJs4COnRIgjDz
3yEdYyEPFlhNlE7/8cTFRy1xFpDbm8GrUxQld/CZsKwECT+f/wNwPeXHrlLoKaBN2vukf6daX7Lr
iBMw6HegDHm+lGlLKTDtLiUAfeajDLigEKJPv1zLCSXTSs0y/jmDNTuKbHcUE55lg808BAa2E7u/
eUtuSgZ6R8JBLwfc1ldGybYUdatIEy4RsAWXrxFGkZGIop+43ibZy6dA+sxzLqPuklaxanSKKfPJ
CcFcA49CQXEIOCa0mf9GT5rPf1yC6fmUZcOCsClq1afLBV8ExA4kNBbaYMXRi5S74UBe4K4zckvs
lhGSBlAEvFMN0HSjzE7rc7Cm+s5rTppEPq+Lpmlm+UoaNF9oGqzZEWp+AzO/cgOt08z4JjSlzQxD
TmXnFInpej7sUAhFpTfaqmGarftk0qw2rvsdLKRjpdkjXSsA9j0vlMYw7lpM6DofK/gyrUHCu++o
7Af5UIVaS/1+fQnd41rF1zcy3e/0JnCvcuRBNUI0QlTFU27y8b6/M8Ujl0u13DDm9MSrrGx2aW4m
izWeTdQxcok8fi3l+rRdplq9TcZ8BAVcGiK0jwtOreCs4amD2iaBcQXw0Y3kCs+N28cxdVoCFUCi
NuQtuEM8l8dMAquP8tDKHW1oGaA+DAXVC9WBPjzqlxc4yAHWQy7clAWg893f2t/N299NnLvSXxwS
uepsCYOEGzF2mwbFHU3pmGfOeFPZF1NwBxTZWLlmUpbGaFSmcW4bJ15UQRGagTVPE7mhTHJ/o9VE
2ZVQwqcJz55+QAhyaYGg1+n+0XkgsqscL4hC4LQwz8m3lOx5EDjvKSYvAtD5lEnmab9M84X0HoTH
XK9Qoma06qhp3SHtrdFKL3ksQ6aUuSq3VejGyF3IVoHyJBIdovFoFikvYWBocv94G0ACPmrBS4a8
uLQWdzYjpt2XjNrGbuswHiLSJhAIC7yms4B2lKAhRohyjwS9CTX5DailG9qnj2lvvLdlM+Ubrzt1
GyPzvMdAhod4TnVPGpCN43iNWsj6zBBD7zuq1bflT6z/sKYyiZShCy/Bo3Bu7gxBFpKIHs+xSVMF
i3WZIZnslKjNzFbgxH4csKJiVhMZFynt5djsvft0c+9+9NTS5w3i98H/vQRdNxkehSmlEtIWEiia
5CUfWNo1eQb5XPn98M7TZUAhZjohaen6cMIFypDit27n8UN8ZCxgbbrNHsWTbW5Ak2RJqXzO2FST
Q6bCJKgl1ZY1I+7j7ApklsQ0XKUULPjo4e/Y8BnmHXrF7a3JMzmlhGvONKRKLSlyHgqua7SV1woa
6TMwOedJcyrciJLX2iqgxQW46oXangmNPVcEKgs0cGGGWUInSKgrk5qOSiTmGDYUAOCYoCvOGDfF
RtRZbAOb6jzojHJ9CiPnUkpt5GXweQBdIOffxc/Q3MkTxB0r75bO1OESGzQMVD0rD/coxshBFLfa
2Gil6c2NNlu+ZDlWnuSiR5v4WOuAlNJPH6t1yXcBh8cIHJQSEjMcX7G1wtt1eBtSU6YKHVu99FPU
dTCWAx0K9icwJlzopy11lG4jmPIssQ1JHylGvqrDh81IYen3TJpNqgi1ERzJ9QYwTpdpDOrGWcEU
5o2d9wTcUkujTajdYe0E3m2LsqYBNo3oWJedUWPbDu3vN1gIN6DDYMvWFLaGg607d3m2M2q5SFPn
FBkwcNgwuFPYJ3lqjI8bS761EdnwO9Pb3GvkIwAB63SSqR9TGpQbjoRmZCOYTBEsXz7Hqg62bzhp
+3B+EJsr6IOZg/PimS9UB9jPElUsYVlM2yjJAIN4RKbTZFBXsIYN56J1nNJljNmCQdvBZj3sXufT
aK6Sde9Wv7wa6MfDHWzQm4YKVqm2DD3L+TN9+DiuThOz19BzvXhqTpcNRJ/+PorXqI3v2OpECJwk
CEZjPVnbAu6kzz0zdec7NRePAXknvMfMAw+X5sCFhO/rM+o4B5pVh9JpiyQPCR1zRg5Yb7gpMt5R
MXHUgPCUz6/0DijBgeC9Q8eb+w68wd0y+Vi/FMKUjkW9hIOzIxQHPOPKCW5Z9+GgU9Ywa1UlCyo5
pA7hO3W4G3sjGXazRUme+zs55NZW0yLGc/XCAN/a//4VP+BSjPquIbtutk4l3uzxd6dpefmhqy6d
5oiZHG1C8PWe1xBJaoHTv3FkfCOUHmJobnM++fvRfT5uE632X5DE5EgWz5su1/eZK75SEiuSAP6g
JkqrZKC6NRTIzAsKjOFQ6GTiMRxqEnHVSVUORDVzbAdaSVZ8q4hKIxiAjkNNS+wSr/Q2EWtqAtk9
f5seQc7rq2lXXd0QHt9Gs6sZu4DrpTc2SPEN2HJ5yqpcB/HhDcDd3s9NlQZwU6qRpD2CU02REtIw
zenwf9VlQRKI9P3BjdFmsOg2e7gypD60xJ1GOpbwfZLc0UZ4huPvhD87BwY7fJ3y9S5d26boYO9+
iLxugss8UDVilMlCeDDPcsaYYVhmUhInNS5liXJ/F6ZwD3XggHmdUjCtM6/ScB1TiWLj4QFbxg03
T0dQdhqQ040mWVV4oQoutROWXMOLlW9ajXQYUxeKFgkfcnUeaWRRr7o87Sn5fHVOIJempZqVZbnZ
nM94Ef6PBr1eAyn3N/EhRhBgg0QhJ2I7DLq+5NbV67t0Y/bdlsYZn4e8XzGuEsEElfsouLL1ZgiW
IoizwuSQc+3z9pkxcAquH02HAHRR5kmiigmq0r0LTldDYW11fJMNnmQ55lOx7L0sjzUfNMpTW5Ov
TtjR4VrNADNVLZQOIqA26JVYpupIgSoUDNPtnx1eByNp9FtV0I++cqpCvRU/yZgKvtWNQSqHwqVi
Y3a+KDcvlQ9sJ2nzSG1CMp+UfW2smTwxjsuVaXqI53Es/PTEDBoFE/hLjNjt0alZZHtG+Eup3c9F
sbGkgD68gjOUqDmjisCXZy+5bMMGeFQK716AKFW5LBigqcLMh2OsAggI4LvNw4ZxwuPRb+lOt43i
+KG2s8huS3dNHaMqxoGfXVX864IOr94Qbvu84eHcLIz1fyIh/h9iT1xUXwADZfvoLzqlaoO5+re+
Gt73VNSCOYUx/hOfqDVzA9eSLPgIRMLVLzrdzeUp1BH38OKOahGNk38M1fPWR1sWrlBaxLX4R/gP
TrfGc3skxET1oZhe4LeilxgGYuDbZhg4C2Sfzid2z39dokzApP0DKpXkcIXsl8UN+DhnlFfuFayl
qgKAAWgVAuCTiKNo0apKh1en/emWDuTR6R6SsEJt1Hyp1xr16j+cKW5UBnN0nVK8raIDLkfKXari
1+UjhBsOe1lIWzUuLLCU9reL3e4FivJQAZnHHTclvbAs2SI4PVfBPOczkv1GlQ/uyWhefNQ+ltOi
ZB1uxPmHwESnedAiMfhZj71GuZg7AG2Gswam4kODr5o5XQlELaoKYXvx/Qn/Kwhafg19viKKalxH
1AbwPRZhg6Ut0tCiVJXdG0PlxWu4nDmsr0p2WXqtKml2oKru1SvwtbgMo63Y/q7s8Mcnhhs0uaDY
/+dJt00H3gbVYbdh7ZTboDzhRs9D67k5xkbw19lEnbdissMQoOAcXeOkFYkPZD0Vprz8Qe/lcffl
MX5vv4Q3VU35bmO+y/ZcnmzN8kWpl8OPH84OT8942lBIlpVTtFXq6IeDo48NX1bH/qBbaTNzzn7R
s84pe6Z2sX+QnXMp9WO6q6rXpTRhQahfug5KK2uNyKFzBlyaAae5yEhgh+Hd7cjsaOka73mfQzfE
RN7dLeBM/NZOSr/dtzYsa6k5LXOkvw+lUy4uI6hC7PSq2EyHng9+en/0JXh/dII2mfNu/5efQLn8
l7egIKpS//Je5hdFkv6iPkZ5ZohBCl+ToeLcM+yq+uFdMQQLwXtKU6p+YdfFlHdHFKA4NU2dqhJn
MMCpFmIEOtaLqSdUODJVJv7RA9/M+vpxxkpDbrQwU7GhSeVdrTlX76glSdCsWncKelUi0oQIJqRO
uvbRC7bZtGRq+F4f4Ww3fnZvuJYGqlCC0sQLbMC3GMNwTVAQYUw93JuqVmHD/bXFo9ka9vtb1GAL
guQf6jVKdyCWg8gWDa/0KFRnLfDUJq85tlQ7saTTZGyac2VxSn8mCwXlzcn5FgQkHVANWwOc60y5
/wW784oSnGkAAA==''')

import base64, gzip
_src = gzip.decompress(base64.b64decode(M45_ABLATION_B64))
with open('/kaggle/working/m45_ablation.py', 'wb') as fh:
    fh.write(_src)
print('wrote m45_ablation.py', len(_src), 'bytes')


wrote m45_ablation.py 27036 bytes


In [5]:
M49_XVAL_B64 = (
'''...''')

import base64, gzip
_src = gzip.decompress(base64.b64decode(M49_XVAL_B64))
with open('/kaggle/working/m49_xval.py', 'wb') as fh:
    fh.write(_src)
print('wrote m49_xval.py', len(_src), 'bytes')


wrote m49_xval.py 69286 bytes


## 4. Self-test on synthetic corpora --- runs before any real data

Builds a fake SPRSound (wav + JSON) and a fake HF_Lung_V1 (wav + `_label.txt`) in a temp
directory and pushes both through the real index builders, the real spectrogram function and
the real network. It checks the things that fail *silently*: that every label string lands in
the right ICBHI class, that an inhalation and its exhalation are paired into one cycle with
the right times, that an adventitious span outside a cycle does not label it, that the two
slices of one session share a bootstrap group, that an unknown label **raises** instead of
being bucketed into Normal, and that the metric being applied is the official one rather than
the inflated macro variant.

In [6]:
sys.path.insert(0, "/kaggle/working")
import m49_xval as X

assert X.selftest() == 0, "self-test failed - do not run the evaluation"


  metric        P3 matrix -> 0.5764 (want 0.5764)
  sprsound      event labels [2, 4, 6, 2] (want [2, 4, 6, 2])
  sprsound      record rows 7 (want 7 - Poor Quality excluded)
  hflung        8 cycles from 8 files (want 8 - one cycle each)
  hflung        I+E paired into [1.0, 3.5] as 'I+E' (want [1.0, 3.5] 'I+E')
  hflung        unpaired I kept: [(1.0, 2.0, 'I'), (5.0, 7.0, 'I+E')] (want I, then I+E)
  hflung        two slices of one session share a group: True (want True)
  unknown label raises as required
  log_mel       shape (1, 128, 801) range [0.00, 1.00] (want (1, 128, 801) inside [0, 1])
  forward       (2, 4) (want (2, 4))
  bootstrap     perfect predictions -> [1.0, 1.0] (want [1.0, 1.0])
  bootstrap     one-class slice -> [None, None] (want [None, None])
  detect-only   Se 0.85 (want 0.85 - cross-type errors now count)
  baseline      always-Normal 0.5 (want 0.5 exactly - Se 0, Sp 1 by construction)
  baseline      prior-matched random 0.3792 (want below always-Normal)
  cal

## 5. The gate --- reproduce the checkpoint's own ICBHI score

If this cell does not reproduce the score stored inside the checkpoint to within 1e-3, every
number after it is uninterpretable and the notebook stops here.

In [7]:
model, cfg, meta = X.load_checkpoint(CKPT)
print("checkpoint:", meta)
print("preprocessing:", {k: cfg[k] for k in
      ("n_mels", "duration_s", "padding", "minmax", "bandpass", "denoise", "ampnorm",
       "pretrained")})

# return_details keeps the 2,636-cycle pass this cell just made. The evaluation below reuses
# it for the calibration comparison and for ICBHI's own class prior, so those come from the
# verified forward pass rather than a second one that could differ.
GATE = X.verify_on_icbhi(model, cfg, meta, ICBHI_AUDIO, ICBHI_SPLIT,
                         tol=0.005, return_details=True, batch_size=64)
ICBHI_REF = GATE["score"]


checkpoint: {'path': 'best_model.pth', 'row': None, 'epoch': 38, 'reported_icbhi_score': 0.5602301973119816, 'total_params': 2263160}
preprocessing: {'n_mels': 128, 'duration_s': 8.0, 'padding': 'wrap', 'minmax': True, 'bandpass': False, 'denoise': False, 'ampnorm': False, 'pretrained': True}
  ICBHI verification: 2636 test cycles, 47 patients
  reproduced 0.5588 | checkpoint reports 0.5602 | tol 0.005
  PASS - forward path reproduces the training run.


## 6. Cycle level --- the headline number

One forward pass over every inhalation+exhalation cycle. Read the printed label vocabulary and
the drop counts before the score: they are the audit that the cycle builder saw what the
corpus actually contains.

In [8]:
doc_cycle = X.run("hflung", HFL_ROOT, CKPT, WORK,
                  icbhi_gate=GATE, limit=SMOKE, batch_size=64, probe=RUN_PROBE)


  checkpoint best_model.pth (row None, epoch 38, reported ICBHI 0.5602)
  preprocessing: n_mels=128 dur=8.0s pad=wrap minmax=True ampnorm=False bandpass=False denoise=False
  in-domain reference 0.5588 from the caller's gate (2636 ICBHI test cycles carried over).
  HF_Lung_V1 35368 cycles from 9765 recordings, 4530 recording groups
    label vocabulary seen: {'D': 15606, 'E': 18349, 'I': 34095, 'Rhonchi': 4740, 'Stridor': 686, 'Wheeze': 8457}
    phase composition: {'I+E': 17067, 'I': 17020, 'E': 1281}  (48.3% are whole I+E cycles)
    dropped: {'no_label_file': 0, 'no_phase_labels': 266, 'short_or_reversed': 9, 'past_eof': 0}
      3200/35368  (72s)
      6400/35368  (145s)
      9600/35368  (217s)
      12800/35368  (289s)
      16000/35368  (360s)
      19200/35368  (431s)
      22400/35368  (503s)
      25600/35368  (574s)
      28800/35368  (644s)
      32000/35368  (716s)
      35200/35368  (787s)
  analysis arms: baselines, strict/broad, prior correction, calibration ...
  froze

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


  M49_HF_Lung_V1_cycle  |  HF_Lung_V1  |  unit: respiratory_cycle (I+E)
  n = 35368 over 4530 recording_groups
  ICBHI official   0.4933 [0.4772, 0.5111]   (recording_group-level CI)
  Se 0.3372   Sp 0.6493   acc 0.4773   macro-F1 0.3704
  detect-only      0.6211  (Se 0.5928, Sp unchanged)
  in-domain ICBHI  0.5588   ->   delta -0.0655   (0.50 = always-Normal)
  segments         median 1.32 s -> tiled 6.04x into the 8 s window (ICBHI median 2.42 s, ~3.3x)
  distribution over    Normal   Crackle    Wheeze      Both
    true              15879     10142      7903      1444
    predicted         18247      4235      9383      3503
  frozen-feature probe (uses target labels): 0.4871 [0.4671, 0.5066]
  wrote /kaggle/working/M49/results_M49_HF_Lung_V1_cycle.json


## Collect

Download `M49_results.zip` from the Output panel, unzip it into `M49_cross_dataset/`, and
commit. The results JSONs follow `Model_Training_Protocol.md` section 4, carry the full
taxonomy mapping in `dataset_info.label_mapping`, and name the bootstrap's grouping unit in
`best_metrics.icbhi_score_official_ci95_unit`.

In [9]:
import shutil
shutil.make_archive("/kaggle/working/M49_results", "zip", WORK)
print("zipped:", os.path.getsize("/kaggle/working/M49_results.zip"), "bytes")
for f in sorted(glob.glob(os.path.join(WORK, "results_M49_*.json"))):
    d = json.load(open(f))
    b, t = d["best_metrics"], d["transfer"]
    print(f"{d['meta']['model_id']:26s} n={d['dataset_info']['test_samples']:6d}  "
          f"ICBHI {X._s(b['icbhi_score_official'])} {b['icbhi_score_official_ci95']}  "
          f"Se {X._s(b['icbhi_se_official'])}  Sp {X._s(b['icbhi_sp_official'])}  "
          f"delta vs in-domain {X._s(t['delta'], sign=True)}")


zipped: 697537 bytes
M49_HF_Lung_V1_cycle       n= 35368  ICBHI 0.4933 [0.4772, 0.5111]  Se 0.3372  Sp 0.6493  delta vs in-domain -0.0655
